# Single guiding-centre orbit with BM4

This experiment evolves one guiding centre in a reproducible periodic potential. The BM4 direct/adjoint composition uses the coupled extended GC formulation, and the animation shows the computed orbit over the gyroaveraged potential.

All potential, physical, initial-condition, formulation, and numerical parameters are explicit below.

In [1]:
%matplotlib inline

import numpy as np

from dynamics import GuidingCenterDynamics
from initial_conditions import TrajectoryGC
from simulation import (
    BM4Composition,
    GCExtendedFormulation,
    InitialValueProblem,
    SimulationRequest,
    SimulationRunner,
)
from studies import RandomPotentialConfig
from visualization import (
    animate_gc_particle_solution,
    display_animation,
)

## Reproducible configuration

`rho` sets the normalized Larmor radius used to gyroaverage the potential. `coupling_frequency` is the harmonic mixing frequency of the doubled-state GC formulation used by BM4.

In [2]:
potential_config = RandomPotentialConfig(
    amplitude=0.7,
    max_wave_number=16,
    nx=48,
    ny=48,
    seed=27,
    interpolation_order=5,
)
potential = potential_config.build()

rho = 0.3
initial_position = (np.pi, np.pi)
coupling_frequency = np.pi / 8

t_span = (0.0, 4 * np.pi)
max_step = 0.01
output_sample_count = 241

trajectory = TrajectoryGC.from_components(
    x=np.asarray([initial_position[0]]),
    y=np.asarray([initial_position[1]]),
    rho=rho,
)
dynamics = GuidingCenterDynamics(potential, rho=rho)
problem = InitialValueProblem(dynamics, trajectory)
method = BM4Composition(
    GCExtendedFormulation(coupling_frequency=coupling_frequency)
)
request = SimulationRequest.uniform(
    t_span=t_span,
    max_step=max_step,
    sample_count=output_sample_count,
)

print(potential_config)
print(f"GC coupling frequency: {coupling_frequency:.6f}")

RandomPotentialConfig(amplitude=0.7, max_wave_number=16, nx=48, ny=48, seed=27, interpolation_order=5)
GC coupling frequency: 0.392699


## BM4 integration

In [3]:
solution = SimulationRunner().simulate(problem, method, request)

print(f"Fixed BM4 steps: {solution.diagnostics['step_count']}")

Fixed BM4 steps: 1257


## Animated guiding-centre orbit

The background is the effective potential used by the GC dynamics, rather than the unaveraged input potential. The black curve is the orbit accumulated up to the current frame.

In [4]:
animation = animate_gc_particle_solution(
    dynamics.effective_potential,
    solution,
    frames=61,
    interval=80,
    cmap="RdBu_r",
    repeat=True,
)

display_animation(animation, embed_limit_mb=30.0)